In [2]:
# ======================================================================
# 00_shared_utils.ipynb
# FinGap-Governance | Load & smoke-test the shared experiment core.
# ----------------------------------------------------------------------
# Run this first. It imports fingap_core (the single source of truth for
# the LLM client, retrieval index, provision-judgment graph, prompts and
# scorer), prints the active configuration, and runs one offline-safe
# sanity check per condition. No API cost in mock mode.
# ======================================================================
from fingap_core import *
import _build_artifacts as _B
_B.write_all()                                   # ensure data/corpus, graph, cases exist
print(banner())

_cases = load_cases("advisory")
_rag   = RagIndex("advisory").build()
print(f"\nsmoke test | advisory cases={len(_cases)} corpus_chunks={len(_rag.chunks)} "
      f"graph_nodes={len(load_graph('advisory').get('provision_nodes', []))}")
for _cond in CONDITIONS:
    _r = run_once(_cases[0], _cond, RUN_TIERS[0], 0, rag=_rag)
    print(f"  {_cond:11} det={_r.determination:15} cite={_r.cited} "
          f"trace={_r.traceable} fail={_r.failure}")
print("\nOK: core loaded. Set OPENAI_API_KEY in ../.env to switch from mock to live.")


FinGap-Governance | MODE=live
models={'weak': 'gpt-4o-mini', 'mid': 'gpt-5.4-mini', 'strong': 'gpt-5.4'} | tiers=['weak', 'mid', 'strong'] | repeats=3 | temp=0.0 | top_k=4 | llm_judge=True

smoke test | advisory cases=18 corpus_chunks=6 graph_nodes=4
  direct      det=suitability     cite=['Article 9 of the Financial Investment Services and Capital Markets Act'] trace=False fail=hallucination
  vanilla_rag det=suitability     cite=['FCPA-Art17#0'] trace=False fail=hallucination
  graph_rag   det=suitability     cite=['FCPA-Art17'] trace=True fail=None

OK: core loaded. Set OPENAI_API_KEY in ../.env to switch from mock to live.
